In [ ]:
from google.colab import files
import os
import shutil
import zipfile

# Install Kaggle
!pip install -q kaggle

# Clean old kaggle credential files
!rm -rf /root/.kaggle
!rm -f kaggle.json
!rm -f kaggle*.json
!rm -f *.zip

# Upload kaggle.json
print("Please upload your NEW kaggle.json file")
uploaded = files.upload()

uploaded_filename = list(uploaded.keys())[0]
print("Uploaded file:", uploaded_filename)

# Setup Kaggle credentials
os.makedirs("/root/.kaggle", exist_ok=True)
shutil.copy(uploaded_filename, "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)

# Check Kaggle login
print("Checking Kaggle API...")
!kaggle datasets list -s gym -n 5

# Dataset slug
dataset_slug = "muhammadtayyabsohail/gym-recommendation-dataset"

# Download dataset
print("Downloading dataset...")
result = os.system(f"kaggle datasets download -d {dataset_slug} -p /content --force")

# Only unzip if download worked
zip_files = [f for f in os.listdir("/content") if f.endswith(".zip")]

if result == 0 and len(zip_files) > 0:
    zip_file = "/content/" + zip_files[0]
    extract_folder = "/content/gym_dataset"

    os.makedirs(extract_folder, exist_ok=True)

    with zipfile.ZipFile(zip_file, "r") as zip_ref:
        zip_ref.extractall(extract_folder)

    print("Dataset downloaded and unzipped successfully!")
    print("Files inside dataset:")
    print(os.listdir(extract_folder))
else:
    print("Dataset download failed.")
    print("Fix: Open the dataset page in Kaggle, login, accept any terms/rules, then try again.")

Please upload your NEW kaggle.json file


Saving kaggle.json to kaggle.json
Uploaded file: kaggle.json
Checking Kaggle API...
usage: kaggle [-h] [-v] [-W]
              {competitions,c,datasets,d,kernels,k,models,m,files,f,benchmarks,b,config,auth}
              ...
kaggle: error: unrecognized arguments: -n 5
Dataset downloaded and unzipped successfully!
Files inside dataset:
['gym recommendation (IMP).csv']


In [ ]:
from google.colab import files as colab_files
import pandas as pd
import numpy as np
import os
import json
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Find CSV file automatically
dataset_folder = "gym_dataset"

csv_files = []
for root, dirs, file_list in os.walk(dataset_folder):
    for file in file_list:
        if file.endswith(".csv"):
            csv_files.append(os.path.join(root, file))

print("CSV files found:", csv_files)

if len(csv_files) == 0:
    raise FileNotFoundError("No CSV file found inside gym_dataset folder")

csv_path = csv_files[0]
print("Using CSV file:", csv_path)

# 2. Load dataset
df = pd.read_csv(csv_path)

# Clean column names
df.columns = df.columns.str.strip()

# Remove useless unnamed columns
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

# 3. Required columns for this dataset
required_columns = [
    "Sex",
    "Age",
    "Height",
    "Weight",
    "Hypertension",
    "Diabetes",
    "BMI",
    "Level",
    "Fitness Goal",
    "Fitness Type",
    "Exercises"
]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing columns in dataset: {missing_columns}")

# 4. Drop empty rows
df = df.dropna(subset=required_columns)

# Optional: make text columns clean
text_columns = ["Sex", "Hypertension", "Diabetes", "Level", "Fitness Goal", "Fitness Type", "Exercises"]

for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

# 5. Features and target
X = df[
    [
        "Sex",
        "Age",
        "Height",
        "Weight",
        "Hypertension",
        "Diabetes",
        "BMI",
        "Level",
        "Fitness Goal",
        "Fitness Type"
    ]
]

y_text = df["Exercises"]

# 6. Encode target output: Exercises
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y_text)

# 7. Split data
# Stratify only if every class has at least 2 samples
class_counts = pd.Series(y).value_counts()

if class_counts.min() >= 2:
    stratify_value = y
else:
    stratify_value = None

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=stratify_value
)

# 8. Preprocessing
numeric_features = ["Age", "Height", "Weight", "BMI"]
categorical_features = ["Sex", "Hypertension", "Diabetes", "Level", "Fitness Goal", "Fitness Type"]

# Make OneHotEncoder compatible with different sklearn versions
try:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", one_hot, categorical_features)
    ]
)

# 9. Full model pipeline
model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced"
        ))
    ]
)

# 10. Train model
model_pipeline.fit(X_train, y_train)

# 11. Test accuracy
y_pred = model_pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", accuracy)

# 12. Save model and target encoder
joblib.dump(model_pipeline, "AI_04_Workout_Planner_Pipeline.joblib")
joblib.dump(target_encoder, "AI_04_Exercises_Target_Encoder.joblib")

# 13. Save input options for FastAPI/frontend
metadata = {
    "input_columns": X.columns.tolist(),
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "sex_options": sorted(df["Sex"].unique().tolist()),
    "hypertension_options": sorted(df["Hypertension"].unique().tolist()),
    "diabetes_options": sorted(df["Diabetes"].unique().tolist()),
    "level_options": sorted(df["Level"].unique().tolist()),
    "fitness_goal_options": sorted(df["Fitness Goal"].unique().tolist()),
    "fitness_type_options": sorted(df["Fitness Type"].unique().tolist()),
    "target_classes_count": len(target_encoder.classes_)
}

with open("AI_04_Workout_Planner_Metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Metadata:")
print(json.dumps(metadata, indent=4))

# 14. Test single prediction example
sample_input = pd.DataFrame([{
    "Sex": "Male",
    "Age": 21,
    "Height": 175,
    "Weight": 95,
    "Hypertension": "No",
    "Diabetes": "No",
    "BMI": 31.0,
    "Level": "Beginner",
    "Fitness Goal": "Weight Loss",
    "Fitness Type": "Cardio Fitness"
}])

prediction_encoded = model_pipeline.predict(sample_input)[0]
prediction_text = target_encoder.inverse_transform([prediction_encoded])[0]

print("Sample Recommended Exercises:")
print(prediction_text)

# 15. Download files
colab_files.download("AI_04_Workout_Planner_Pipeline.joblib")
colab_files.download("AI_04_Exercises_Target_Encoder.joblib")
colab_files.download("AI_04_Workout_Planner_Metadata.json")

CSV files found: ['gym_dataset/gym recommendation (IMP).csv']
Using CSV file: gym_dataset/gym recommendation (IMP).csv
Dataset shape: (14568, 13)
Columns: ['ID', 'Sex', 'Age', 'Height', 'Weight', 'Hypertension', 'Diabetes', 'BMI', 'Level', 'Fitness Goal', 'Fitness Type', 'Exercises', 'Diet']
Model Accuracy: 0.9979409746053535
Metadata:
{
    "input_columns": [
        "Sex",
        "Age",
        "Height",
        "Weight",
        "Hypertension",
        "Diabetes",
        "BMI",
        "Level",
        "Fitness Goal",
        "Fitness Type"
    ],
    "numeric_features": [
        "Age",
        "Height",
        "Weight",
        "BMI"
    ],
    "categorical_features": [
        "Sex",
        "Hypertension",
        "Diabetes",
        "Level",
        "Fitness Goal",
        "Fitness Type"
    ],
    "sex_options": [
        "Female",
        "Male"
    ],
    "hypertension_options": [
        "No",
        "Yes"
    ],
    "diabetes_options": [
        "No",
        "Yes"
   

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>